# Mango Leaf Disease Classification

This notebook builds a full image-classification workflow for the Kaggle mango leaf disease dataset. It compares handcrafted features and deep features across four classifiers:

- Neural Network / CNN
- k-Nearest Neighbors
- Gaussian Naive Bayes
- Support Vector Machine with RBF kernel

## Workflow

This notebook implements a comprehensive machine learning pipeline:

1. **Dataset Discovery** - Automatically find and load labeled images
2. **Exploratory Analysis** - Visualize class distribution and sample images
3. **Data Splitting** - Stratified train/validation/test splits
4. **Feature Engineering** - Extract handcrafted features:
   - RGB color histogram
   - HSV color histogram
   - GLCM texture features
   - Shape descriptors (area, perimeter, solidity, etc.)
5. **Classical ML Models** - Train on handcrafted features:
   - k-Nearest Neighbors
   - Gaussian Naive Bayes
   - Support Vector Machine (RBF kernel)
6. **Deep Features** - Extract MobileNetV2 embeddings and re-train models
7. **CNN Model** - Build and train a convolutional neural network with augmentation
8. **Comprehensive Evaluation**:
   - Accuracy, precision, recall, F1-score
   - Confusion matrices for all models
   - ROC curves
   - Performance comparison charts
9. **Model Persistence** - Save trained models and scalers to `models/` folder
10. **Report Generation** - Generate markdown report with insights

### Key Insights to Expect

- CNN typically achieves **highest accuracy** on image classification
- Handcrafted features are **faster** but less accurate
- Deep features provide **better transfer learning** than raw pixels
- Model choice depends on **accuracy vs. speed tradeoff**


In [3]:

# ============================================================================
# SETUP: Import libraries and configure project environment
# ============================================================================

from pathlib import Path
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Project modules
from src.config import (
    RAW_DATA_DIR, RESULTS_DIR, MODELS_DIR, IMAGE_SIZE,
    RANDOM_STATE, TEST_SIZE, VALIDATION_SIZE, BATCH_SIZE, EPOCHS,
    KNN_NEIGHBORS, SVM_C
)
from src.utils import set_global_seed, ensure_directory, discover_images, stratified_split
from src.features import (
    load_image, extract_handcrafted_matrix,
    build_deep_feature_extractor, extract_deep_features
)
from src.models import (
    train_knn, train_gaussian_nb, train_svm_rbf,
    evaluate_classifier, build_cnn_model, EvaluationResult
)
from src.visualization import (
    plot_class_distribution, plot_sample_images, plot_confusion_matrix,
    plot_metric_comparison, plot_training_curves, plot_roc_curves
)

# Configuration
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")
set_global_seed(RANDOM_STATE)
ensure_directory(RESULTS_DIR)
ensure_directory(MODELS_DIR)

# Print configuration
print("=" * 70)
print("MANGO LEAF DISEASE CLASSIFICATION - CONFIGURATION")
print("=" * 70)
print(f"Project root: {RAW_DATA_DIR.parent}")
print(f"Image size: {IMAGE_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"CNN epochs: {EPOCHS}")
print(f"Train/Val/Test split: {1-TEST_SIZE-VALIDATION_SIZE:.1%} / {VALIDATION_SIZE:.1%} / {TEST_SIZE:.1%}")
print(f"Random seed: {RANDOM_STATE}")
print("=" * 70)


ModuleNotFoundError: No module named 'src'

In [ ]:

# ============================================================================
# STEP 1: DATASET DISCOVERY AND LOADING
# ============================================================================

print("\nLoading dataset...")
print("-" * 70)

# Try multiple potential dataset roots (support flexible folder structure)
candidate_roots = [
    RAW_DATA_DIR / "mango_leaf_disease_dataset",  # Kaggle default extraction
    RAW_DATA_DIR,  # Direct class folders in data/raw/
]

dataframe = pd.DataFrame()
dataset_root = None

for root in candidate_roots:
    if root.exists():
        candidate = discover_images(root)
        if not candidate.empty:
            dataframe = candidate
            dataset_root = root
            break

if dataframe.empty:
    raise FileNotFoundError(
        "No images found!\n"
        "Expected structure:\n"
        "  data/raw/mango_leaf_disease_dataset/\n"
        "    ├── class_1/\n"
        "    │   ├── image1.jpg\n"
        "    │   └── ...\n"
        "    └── class_2/\n"
        "\n"
        "Download from: https://www.kaggle.com/datasets/warcoder/mango-leaf-disease-dataset\n"
        "Or run: python setup_dataset.py"
    )

print(f"Dataset root: {dataset_root}")
print(f"Total images discovered: {len(dataframe)}")
print(f"Unique classes: {dataframe['label'].nunique()}")
print("\nClass distribution:")
print(dataframe['label'].value_counts().to_string())
print("-" * 70)


In [ ]:

# ============================================================================
# STEP 2: EXPLORATORY DATA ANALYSIS
# ============================================================================

print("\nGenerating exploratory visualizations...")

# Plot class distribution
print("  • Class distribution...", end=" ")
plot_class_distribution(dataframe, output_path=RESULTS_DIR / "class_distribution.png")
print("✓")

# Plot sample images from each class
print("  • Sample images...", end=" ")
samples = []
for label in dataframe['label'].value_counts().index[:9]:
    sample_row = dataframe[dataframe['label'] == label].iloc[0]
    image = load_image(sample_row['image_path'], image_size=IMAGE_SIZE)
    samples.append((image, label))
plot_sample_images(samples, output_path=RESULTS_DIR / "sample_images.png")
print("✓")

print("✓ Exploratory analysis complete")


In [ ]:

# ============================================================================
# STEP 3: DATA PREPARATION
# ============================================================================

print("\nPreparing data splits...")
print("-" * 70)

# Encode class labels
label_encoder = LabelEncoder()
dataframe['label_id'] = label_encoder.fit_transform(dataframe['label'])
class_names = list(label_encoder.classes_)

print(f"Classes: {class_names}")
print(f"Number of classes: {len(class_names)}")

# Stratified split: maintain class distribution across splits
train_df, val_df, test_df = stratified_split(
    dataframe,
    label_column='label',
    test_size=TEST_SIZE,
    validation_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
)

# Extract image paths and labels
x_train_paths = train_df['image_path'].tolist()
x_val_paths = val_df['image_path'].tolist()
x_test_paths = test_df['image_path'].tolist()
y_train = train_df['label_id'].to_numpy()
y_val = val_df['label_id'].to_numpy()
y_test = test_df['label_id'].to_numpy()

print(f"\nData split:")
print(f"  Train: {len(train_df)} images")
print(f"  Validation: {len(val_df)} images")
print(f"  Test: {len(test_df)} images")
print("-" * 70)


In [ ]:

# ============================================================================
# STEP 4A: HANDCRAFTED FEATURES + CLASSICAL ML MODELS
# ============================================================================

print("\nExtracting handcrafted features...")
print("-" * 70)

# Extract features from all image sets
print("  Train set...", end=" ", flush=True)
x_train_handcrafted = extract_handcrafted_matrix(x_train_paths)
print(f"✓ ({x_train_handcrafted.shape})")

print("  Validation set...", end=" ", flush=True)
x_val_handcrafted = extract_handcrafted_matrix(x_val_paths)
print(f"✓ ({x_val_handcrafted.shape})")

print("  Test set...", end=" ", flush=True)
x_test_handcrafted = extract_handcrafted_matrix(x_test_paths)
print(f"✓ ({x_test_handcrafted.shape})")

# Normalize features
print("\nNormalizing features...")
handcrafted_scaler = StandardScaler()
x_train_scaled = handcrafted_scaler.fit_transform(x_train_handcrafted)
x_val_scaled = handcrafted_scaler.transform(x_val_handcrafted)
x_test_scaled = handcrafted_scaler.transform(x_test_handcrafted)
print("✓ StandardScaler applied")

# Train classical models
print("\nTraining classical ML models on handcrafted features...")
classical_models = {
    'kNN': train_knn(x_train_scaled, y_train, n_neighbors=KNN_NEIGHBORS),
    'Naive Bayes': train_gaussian_nb(x_train_scaled, y_train),
    'SVM (RBF)': train_svm_rbf(x_train_scaled, y_train, c_value=SVM_C),
}

# Evaluate on test set
print("\nEvaluating classical models...")
classical_results = []
classical_predictions = {}
classical_probabilities = {}

for model_name, model in classical_models.items():
    result, predictions, probabilities = evaluate_classifier(
        model, x_test_scaled, y_test, class_count=len(class_names)
    )
    result.model_name = f"{model_name} (handcrafted)"
    classical_results.append(result.as_dict())
    classical_predictions[result.model_name] = predictions
    classical_probabilities[result.model_name] = probabilities
    print(f"  {model_name}: Accuracy={result.accuracy:.4f}, F1={result.f1:.4f}")

classical_results_df = pd.DataFrame(classical_results)
print("\nHandcrafted Feature Results:")
print(classical_results_df.to_string(index=False))
print("-" * 70)


In [ ]:

# ============================================================================
# STEP 4B: DEEP FEATURES + CLASSICAL ML MODELS
# ============================================================================

print("\nExtracting deep features using MobileNetV2...")
print("-" * 70)
print("Note: First run downloads pretrained model (~90MB)")

# Build deep feature extractor
deep_extractor = build_deep_feature_extractor('MobileNetV2', image_size=IMAGE_SIZE)

# Extract features with batching for efficiency
print("  Train set...", end=" ", flush=True)
x_train_deep = extract_deep_features(
    x_train_paths, deep_extractor, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)
print(f"✓ ({x_train_deep.shape})")

print("  Validation set...", end=" ", flush=True)
x_val_deep = extract_deep_features(
    x_val_paths, deep_extractor, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)
print(f"✓ ({x_val_deep.shape})")

print("  Test set...", end=" ", flush=True)
x_test_deep = extract_deep_features(
    x_test_paths, deep_extractor, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE
)
print(f"✓ ({x_test_deep.shape})")

# Normalize deep features
print("\nNormalizing deep features...")
deep_scaler = StandardScaler()
x_train_deep_scaled = deep_scaler.fit_transform(x_train_deep)
x_val_deep_scaled = deep_scaler.transform(x_val_deep)
x_test_deep_scaled = deep_scaler.transform(x_test_deep)
print("✓ StandardScaler applied")

# Train classical models on deep features
print("\nTraining classical ML models on deep features...")
deep_feature_models = {
    'kNN': train_knn(x_train_deep_scaled, y_train, n_neighbors=KNN_NEIGHBORS),
    'Naive Bayes': train_gaussian_nb(x_train_deep_scaled, y_train),
    'SVM (RBF)': train_svm_rbf(x_train_deep_scaled, y_train, c_value=SVM_C),
}

# Evaluate on test set
print("\nEvaluating classical models with deep features...")
deep_feature_results = []
for model_name, model in deep_feature_models.items():
    result, predictions, probabilities = evaluate_classifier(
        model, x_test_deep_scaled, y_test, class_count=len(class_names)
    )
    result.model_name = f"{model_name} (deep features)"
    deep_feature_results.append(result.as_dict())
    print(f"  {model_name}: Accuracy={result.accuracy:.4f}, F1={result.f1:.4f}")

deep_feature_results_df = pd.DataFrame(deep_feature_results)
print("\nDeep Feature Results:")
print(deep_feature_results_df.to_string(index=False))
print("-" * 70)


In [ ]:

# ============================================================================
# STEP 5: CONVOLUTIONAL NEURAL NETWORK (CNN) TRAINING
# ============================================================================

print("\nPreparing CNN with data augmentation...")
print("-" * 70)

import tensorflow as tf

# Create data generators with augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
)
eval_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

# Create data flows from dataframe
train_flow = train_datagen.flow_from_dataframe(
    train_df,
    x_col='image_path',
    y_col='label',
    target_size=IMAGE_SIZE,
    class_mode='sparse',
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=RANDOM_STATE,
)
val_flow = eval_datagen.flow_from_dataframe(
    val_df,
    x_col='image_path',
    y_col='label',
    target_size=IMAGE_SIZE,
    class_mode='sparse',
    batch_size=BATCH_SIZE,
    shuffle=False,
)
test_flow = eval_datagen.flow_from_dataframe(
    test_df,
    x_col='image_path',
    y_col='label',
    target_size=IMAGE_SIZE,
    class_mode='sparse',
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("✓ Data generators created")

# Build CNN model
print("\nBuilding CNN model...")
cnn_model = build_cnn_model((*IMAGE_SIZE, 3), len(class_names))
cnn_model.summary()

# Define callbacks for training
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
]

# Train the model
print("\nTraining CNN (this may take several minutes)...")
print("-" * 70)
history = cnn_model.fit(
    train_flow,
    validation_data=val_flow,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

print("\n" + "=" * 70)
print("Training complete!")
print("=" * 70)

# Plot training curves
plot_training_curves(history, output_path=RESULTS_DIR / 'cnn_training_curves.png')

# Generate predictions on test set
print("\nGenerating CNN predictions on test set...")
cnn_probabilities = cnn_model.predict(test_flow, verbose=0)
cnn_predictions = np.argmax(cnn_probabilities, axis=1)
print("✓ Predictions complete")


In [ ]:

# ============================================================================
# STEP 6: COMPREHENSIVE EVALUATION & MODEL COMPARISON
# ============================================================================

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("\nEvaluating CNN performance...")
print("-" * 70)

# Compute CNN metrics using scikit-learn for consistency with other models
cnn_result = EvaluationResult(
    model_name='CNN',
    accuracy=float(accuracy_score(y_test, cnn_predictions)),
    precision=float(precision_score(y_test, cnn_predictions, average='weighted', zero_division=0)),
    recall=float(recall_score(y_test, cnn_predictions, average='weighted', zero_division=0)),
    f1=float(f1_score(y_test, cnn_predictions, average='weighted', zero_division=0)),
    roc_auc=None,
)

print(f"CNN Performance:")
print(f"  Accuracy: {cnn_result.accuracy:.4f}")
print(f"  Precision: {cnn_result.precision:.4f}")
print(f"  Recall: {cnn_result.recall:.4f}")
print(f"  F1-Score: {cnn_result.f1:.4f}")

# Generate confusion matrices for all models
print("\nGenerating confusion matrices...")
plot_confusion_matrix(
    y_test, cnn_predictions, class_names,
    title='CNN Confusion Matrix',
    output_path=RESULTS_DIR / 'cnn_confusion_matrix.png'
)

for model_name, predictions in classical_predictions.items():
    plot_confusion_matrix(
        y_test, predictions, class_names,
        title=f'{model_name} Confusion Matrix',
        output_path=RESULTS_DIR / f"{model_name.replace(' ', '_').replace('(', '').replace(')', '').lower()}_confusion_matrix.png",
    )

print("✓ Confusion matrices saved")

# Combine all results
print("\nCombining model comparison...")
all_results_df = pd.concat(
    [classical_results_df, deep_feature_results_df, pd.DataFrame([cnn_result.as_dict()])],
    ignore_index=True
)
all_results_df = all_results_df.sort_values(by='f1', ascending=False).reset_index(drop=True)

# Save comparison results
all_results_df.to_csv(RESULTS_DIR / 'model_comparison.csv', index=False)
print(f"✓ Results saved to: {RESULTS_DIR / 'model_comparison.csv'}")

# Generate comparison visualizations
print("\nGenerating comparison visualizations...")
plot_metric_comparison(all_results_df, output_path=RESULTS_DIR / 'model_comparison.png')
print("✓ Comparison chart saved")

# Generate ROC curves
print("\nGenerating ROC curves...")
for model_name, probabilities in classical_probabilities.items():
    if probabilities is not None:
        plot_roc_curves(
            y_test, probabilities, class_names,
            title=f'{model_name} ROC Curves',
            output_path=RESULTS_DIR / f"{model_name.replace(' ', '_').replace('(', '').replace(')', '').lower()}_roc.png",
        )

plot_roc_curves(
    y_test, cnn_probabilities, class_names,
    title='CNN ROC Curves',
    output_path=RESULTS_DIR / 'cnn_roc.png'
)
print("✓ ROC curves saved")

# Save trained models
print("\nSaving trained models...")
joblib.dump(label_encoder, MODELS_DIR / 'label_encoder.joblib')
joblib.dump(handcrafted_scaler, MODELS_DIR / 'handcrafted_scaler.joblib')
joblib.dump(deep_scaler, MODELS_DIR / 'deep_feature_scaler.joblib')
joblib.dump(classical_models['SVM (RBF)'], MODELS_DIR / 'svm_handcrafted.joblib')
joblib.dump(deep_feature_models['SVM (RBF)'], MODELS_DIR / 'svm_deep_features.joblib')
cnn_model.save(MODELS_DIR / 'mango_leaf_cnn.keras')
print("✓ Models saved to: {MODELS_DIR}/")

print("\n" + "=" * 70)
print("FINAL MODEL RANKING")
print("=" * 70)


In [ ]:

print(all_results_df.to_string(index=False))
print("=" * 70)

best_model_name = all_results_df.iloc[0]['model']
best_model_accuracy = all_results_df.iloc[0]['accuracy']
print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"   Accuracy: {best_model_accuracy:.2%}")
print("\nTo generate detailed insights, run:")
print("  python generate_report.py")
print("\nTo make predictions on new images, run:")
print("  python evaluate_models.py --image path/to/leaf.jpg")


## Analysis & Insights

### Key Findings

The above results demonstrate the complete machine learning pipeline for mango leaf disease classification. Here are the key insights:

#### 1. **Model Performance Comparison**
- **CNN typically achieves highest accuracy** by learning task-specific features
- **SVM on deep features** provides excellent performance with lower inference cost
- **Classical models on handcrafted features** are fast but may miss subtle disease patterns

#### 2. **Feature Engineering Impact**
- **Handcrafted features** (~192 dimensions): Fast extraction, domain knowledge required
- **Deep features** (~1280 dimensions): Better generalization, transfer learning benefit
- **End-to-end CNN**: Optimal features learned specifically for this task

#### 3. **Computational Tradeoffs**

| Aspect | Handcrafted | Deep Features | CNN |
|--------|------------|---------------|-----|
| Training Time | Fast | Medium | Very Slow |
| Inference Speed | Very Fast | Medium | Medium |
| Accuracy | Moderate | Good | **Best** |
| Explainability | High | Medium | Low |
| Data Required | Moderate | Moderate-High | High |

#### 4. **Recommendations**

**For Production:**
- Use the best-performing model from the ranking above
- Implement ensemble methods if higher accuracy needed
- Deploy as REST API for scalability

**For Real-time Applications:**
- If latency < 100ms required: Use SVM on handcrafted features
- If latency < 500ms: Use SVM on deep features
- If latency flexible: Use CNN for maximum accuracy

**For Future Improvement:**
- Collect more labeled data to improve CNN generalization
- Try transfer learning with larger models (ResNet, EfficientNet)
- Implement model ensemble combining multiple architectures
- Add explainability using GradCAM for disease localization

### Output Files Generated

All results saved to `results/` folder:
- ✓ Model comparison CSV
- ✓ Confusion matrices (PNG)
- ✓ Training curves (PNG)
- ✓ ROC curves (PNG)
- ✓ Comparison bar charts (PNG)

### Next Steps

1. **Review Results**: Open `results/model_comparison.csv`
2. **Generate Report**: Run `python generate_report.py`
3. **Test on New Images**: Run `python evaluate_models.py --image path/to/leaf.jpg`
4. **Deploy Model**: Use scripts in `models/` folder

### Notes for Report Writing

When writing your analysis report, explain:

1. **Why CNN often outperforms classical ML on images:**
   - Automatic hierarchical feature learning
   - Spatial information preservation
   - Non-linear capacity

2. **When to use each approach:**
   - Handcrafted features: fast, interpretable, limited accuracy
   - Deep features: balanced, better accuracy
   - CNN: maximum accuracy, high computational cost

3. **Agricultural Application:**
   - Early disease detection improves yields
   - Faster than manual inspection
   - Can be deployed on smartphones
   - Reduces fungicide usage through targeted treatment


## Report Notes

Use the ranking table above to explain the final recommendation in your report. A strong report should answer:

- Why the CNN may outperform classical models on raw images
- Why SVM often performs well on smaller feature spaces
- Why Naive Bayes can lag when image features are correlated
- Which handcrafted feature family is strongest: RGB, HSV, texture, or shape descriptors
- How much computation the deep model needs compared with feature-based classifiers

The exported CSV and plots in `results/` can be inserted directly into the final project report.